# coordination-traces-100: Murphy Decomposition Example

Load the per-config traces and reproduce the paper's Table 1 Brier/Alpha/Murphy decomposition.

Requires: numpy, pandas (pip install numpy pandas)

In [ ]:
import json, numpy as np, pandas as pd
from pathlib import Path

CONFIGS = [
    "independent_ensemble", "peer_critique_debate",
    "orchestrator_specialist", "sequential_pipeline", "consensus_alignment",
]

def load_config(name):
    rows = []
    with open(f"data/traces/{name}.jsonl") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

def brier(rows):
    errs = [(r["probability"] - r["outcome"])**2 for r in rows if r["outcome"] is not None]
    return float(np.mean(errs))

def alpha(rows):
    market_brier = np.mean([(r["baseline"] - r["outcome"])**2
                            for r in rows if r["outcome"] is not None and r["baseline"] is not None])
    return float(market_brier - brier(rows))

results = []
for cfg in CONFIGS:
    rows = load_config(cfg)
    results.append({"config": cfg, "n": len(rows),
                    "brier": round(brier(rows), 4),
                    "alpha": round(alpha(rows), 4)})

df = pd.DataFrame(results).sort_values("brier")
print(df.to_string(index=False))

## Compare with published leaderboard


In [ ]:
published = pd.read_csv("data/leaderboard.csv")
print(published[["config","brier","alpha","REL","RES"]].to_string(index=False))